In [1]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.landesdatenbank.nrw.de/
df_35a = pd.read_csv("../data/raw/2024/22811-01i.csv", sep=";", encoding="latin1", skiprows=3)

# Überblick:
print(df_35a.head())
print(df_35a.info())

  Sozialberichterstattung in der amtlichen Statistik  \
0                   Mindestsicherungsquote (Prozent)   
1                                                NaN   
2                                                NaN   
3                                                 05   
4                                                051   

                       Unnamed: 1 Unnamed: 2  
0                             NaN        NaN  
1                             NaN       Jahr  
2                             NaN       2024  
3             Nordrhein-Westfalen       11,1  
4    Düsseldorf, Regierungsbezirk       12,5  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441 entries, 0 to 440
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   Sozialberichterstattung in der amtlichen Statistik  439 non-null    object
 1   Unnamed: 1                      

In [2]:
#relevante Spalten filtern
df = df_35a[['Unnamed: 1','Unnamed: 2']]
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df)

                          Unnamed: 1 Unnamed: 2
1                                NaN       Jahr
2                                NaN       2024
3                Nordrhein-Westfalen       11,1
4       Düsseldorf, Regierungsbezirk       12,5
5            Düsseldorf, krfr. Stadt       11,0
..                               ...        ...
432                  Schwerte, Stadt        9,1
433                      Selm, Stadt        9,5
434                      Unna, Stadt        9,0
435                     Werne, Stadt        6,1
436        Wohnort außerhalb von NRW          x

[436 rows x 2 columns]


In [3]:
# Spalten umbenennen
df = df.rename(columns={"Unnamed: 1": "Name", "Unnamed: 2": "Mindestsicherungsquote"})

# nach Kreisen und kreisfreien Städten filtern und Strings anpassen
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]
df["Name"] = df["Name"].str.strip()

df["Mindestsicherungsquote"] = (
    df["Mindestsicherungsquote"]
    .astype(str)                
    .str.replace(",", ".", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)


print(df)

                                            Name  Mindestsicherungsquote
5                        Düsseldorf, krfr. Stadt                    11.0
6                          Duisburg, krfr. Stadt                    16.8
7                             Essen, krfr. Stadt                    17.7
8                           Krefeld, krfr. Stadt                    14.0
9                   Mönchengladbach, krfr. Stadt                    15.8
10              Mülheim an der Ruhr, krfr. Stadt                    13.9
11                       Oberhausen, krfr. Stadt                    14.5
12                        Remscheid, krfr. Stadt                    12.3
13                         Solingen, krfr. Stadt                    11.4
14                        Wuppertal, krfr. Stadt                    15.7
15                                  Kleve, Kreis                     8.2
32                               Mettmann, Kreis                    10.3
43                             Rhein-Kreis Neuss   

In [4]:
# Zeile löschen und Name anpassen
df = df[df["Name"] != "Aachen, krfr. Stadt (ab 21.10.2009)"]
df = df[df["Name"] != "Essen, krfr. Stadt"]

# fehlende Werte anzeigen
print(df[df["Mindestsicherungsquote"].isna()])

Empty DataFrame
Columns: [Name, Mindestsicherungsquote]
Index: []


In [5]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df
# Tabelle nach Kreis/Stadt sortieren
df = clean_and_sort(df, "Mindestsicherungsquote")
validate_df(
    df,
    not_null=["Mindestsicherungsquote"],
    non_negative=["Mindestsicherungsquote"],
    numeric=["Mindestsicherungsquote"],
    bounds={"Mindestsicherungsquote": (0,100)},
    key_cols=["Name"],
)

DataFrame: alle Checks bestanden


[]

In [6]:
# Übersicht
df["Mindestsicherungsquote"].describe()

count    52.000000
mean     10.709615
std       3.458055
min       6.100000
25%       8.200000
50%       9.950000
75%      12.925000
max      21.700000
Name: Mindestsicherungsquote, dtype: float64

In [7]:
# speichern
df.to_csv("../data/processed/mindestsicherungsquote_2024.csv", index=False)